# Data Description
- Background: Unit 1.7 補充資料驗證：歷史村里界。提供機關：內政部地政司／國土測繪中心；官方來源：https://data.gov.tw/dataset/130549。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/113村里界.zip`；22,267,184 bytes；SHA-256 `32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94`。
  - `data/raw/114村里界.zip`；22,200,490 bytes；SHA-256 `08d2feeced98473215703defdbe9da8f8d551f2fb221029557b813e1a3dd8845`。
- Unit of observation: 村里面（另有三合里附層，不自動與主層合併）。
- Sample period: 官方113／114年版；內部名稱1131128／11401031；後者日期字串不作強制日曆解讀，非全年逐日有效界證明；發布／更新：資料集2020-09-21；各版次實際發布日另未認證。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 主圖層與獨立Sanhe層分開；以既有crosswalk人口官方碼連接，不截碼或重開糖廍爭議。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='historical_boundaries'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,historical_boundaries,data/raw/113村里界.zip,22267184,32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94,present
1,historical_boundaries,data/raw/114村里界.zip,22200490,08d2feeced98473215703defdbe9da8f8d551f2fb221029557b813e1a3dd8845,present


**讀取範圍與CRS／編碼**

,path,member,total_features,loaded_rows,crs,types,bounds,invalid,null_geometry,empty
0,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,7953,7953,EPSG:3824,"{'Polygon': 7866, 'MultiPolygon': 87}","[114.359282, 10.371348, 124.561158, 26.385275]",0,0,0
1,data/raw/113村里界.zip,村里界歷史圖資_113/Village_Sanhe.shp,1,1,"GEOGCS[""GCS_TWD97_2020"",DATUM[""Taiwan_Datum_1997"",SPHEROID[""GRS 1980"",6378137,298.257222101,AUTHORITY[""EPSG"",""7019""]],AUTHORITY[""EPSG"",""1026""]],PRIMEM[""Gree...",{'Polygon': 1},"[120.588067, 22.692684, 120.628476, 22.717004]",0,0,0
2,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp,7956,7956,EPSG:3824,"{'Polygon': 7869, 'MultiPolygon': 87}","[114.359282, 10.371348, 124.561158, 26.385275]",0,0,0
3,data/raw/114村里界.zip,村里界歷史圖資_114/Village_Sanhe.shp,1,1,EPSG:3824,{'Polygon': 1},"[120.588067, 22.692684, 120.628476, 22.717004]",0,0,0


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/113村里界.zip::村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp**

,VILLCODE,COUNTYNAME,TOWNNAME,VILLNAME,VILLENG,COUNTYID,COUNTYCODE,TOWNID,TOWNCODE,NOTE,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,
0,10013030S01,屏東縣,東港鎮,NaN,NaN,T,10013,T03,10013030,未編定村里,1,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
1,64000130006,高雄市,林園區,中門里,Zhongmen Vil.,E,64000,E13,64000130,NaN,2,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
2,64000130008,高雄市,林園區,港埔里,Gangpu Vil.,E,64000,E13,64000130,NaN,3,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
3,64000100010,高雄市,旗津區,上竹里,Shangzhu Vil.,E,64000,E10,64000100,NaN,4,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
4,09007010006,連江縣,南竿鄉,津沙村,Jinsha Vil.,Z,09007,Z01,09007010,NaN,5,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
5,09007010007,連江縣,南竿鄉,馬祖村,Matsu Vil.,Z,09007,Z01,09007010,NaN,6,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
6,09007010009,連江縣,南竿鄉,四維村,Siwei Vil.,Z,09007,Z01,09007010,NaN,7,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
7,09007020002,連江縣,北竿鄉,后沃村,Houwo Vil.,Z,09007,Z02,09007020,NaN,8,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp
8,10013020003,屏東縣,潮州鎮,新榮里,Xinrong Vil.,T,10013,T02,10013020,NaN,9,data/raw/113村里界.zip,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp


**data/raw/113村里界.zip::村里界歷史圖資_113/Village_Sanhe.shp**

,VILLCODE,COUNTYNAME,TOWNNAME,VILLNAME,VILLENG,COUNTYID,COUNTYCODE,TOWNID,TOWNCODE,NOTE,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,
0,10013280006,屏東縣,瑪家鄉,三和村,Sanhe Vil.,T,10013,T28,10013280,None,1,data/raw/113村里界.zip,村里界歷史圖資_113/Village_Sanhe.shp


**data/raw/114村里界.zip::村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp**

,VILLCODE,COUNTYNAME,TOWNNAME,VILLNAME,VILLENG,COUNTYID,COUNTYCODE,TOWNID,TOWNCODE,NOTE,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,
0,65000050015,新北市,新莊區,西盛里,Xisheng Vil.,F,65000,F01,65000050,NaN,1,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
1,65000050055,新北市,新莊區,四維里,Siwei Vil.,F,65000,F01,65000050,NaN,2,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
2,65000050027,新北市,新莊區,南港里,Nangang Vil.,F,65000,F01,65000050,NaN,3,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
3,65000050066,新北市,新莊區,萬安里,Wan'an Vil.,F,65000,F01,65000050,NaN,4,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
4,65000050070,新北市,新莊區,建安里,Jian'an Vil.,F,65000,F01,65000050,NaN,5,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
5,65000050024,新北市,新莊區,富國里,Fuguo Vil.,F,65000,F01,65000050,NaN,6,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
6,65000050078,新北市,新莊區,富民里,Fumin Vil.,F,65000,F01,65000050,NaN,7,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
7,65000050046,新北市,新莊區,合鳳里,Hefeng Vil.,F,65000,F01,65000050,NaN,8,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp
8,65000050018,新北市,新莊區,瓊林里,Qionglin Vil.,F,65000,F01,65000050,NaN,9,data/raw/114村里界.zip,村里界歷史圖資_114/VILLAGE_NLSC_11401031.shp


**data/raw/114村里界.zip::村里界歷史圖資_114/Village_Sanhe.shp**

,VILLCODE,COUNTYNAME,TOWNNAME,VILLNAME,VILLENG,COUNTYID,COUNTYCODE,TOWNID,TOWNCODE,NOTE,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,
0,10013280006,屏東縣,瑪家鄉,三和村,Sanhe Vil.,T,10013,T28,10013280,None,1,data/raw/114村里界.zip,村里界歷史圖資_114/Village_Sanhe.shp


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,歷史村里界
provider,內政部地政司／國土測繪中心
url,https://data.gov.tw/dataset/130549
publication,資料集2020-09-21；各版次實際發布日另未認證
period,官方113／114年版；內部名稱1131128／11401031；後者日期字串不作強制日曆解讀，非全年逐日有效界證明
limitation,主圖層與獨立Sanhe層分開；以既有crosswalk人口官方碼連接，不截碼或重開糖廍爭議
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,historical_boundaries,data/raw/113村里界.zip,22267184,32d7f1d611f91b1c63bea7290dd4db93b919bf380cce92b634fca992c974dd94,present


**ZIP原始成員清單；非解壓暫存資料來源**

,member,bytes,compressed,crc32,zip_datetime
0,村里界歷史圖資_113/,0,0,00000000,"[2026, 8, 27, 15, 4, 38]"
1,村里界歷史圖資_113/VILLAGE_NLSC_1131128.cpg,5,7,0e813c50,"[2026, 8, 27, 14, 35, 48]"
2,村里界歷史圖資_113/VILLAGE_NLSC_1131128.dbf,1447800,145538,5a7e5aa7,"[2026, 8, 27, 14, 35, 48]"
3,村里界歷史圖資_113/VILLAGE_NLSC_1131128.prj,145,125,fe828af0,"[2026, 8, 27, 14, 35, 48]"
4,村里界歷史圖資_113/VILLAGE_NLSC_1131128.qmd,687,340,1b871eba,"[2026, 8, 27, 14, 35, 48]"
5,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shp,33103956,22074757,720ea753,"[2026, 8, 27, 14, 35, 48]"
6,村里界歷史圖資_113/VILLAGE_NLSC_1131128.shx,63724,42058,59b73956,"[2026, 8, 27, 14, 35, 48]"
7,村里界歷史圖資_113/Village_Sanhe.cpg,5,7,0e813c50,"[2026, 8, 27, 14, 42, 26]"
8,村里界歷史圖資_113/Village_Sanhe.dbf,536,186,bc1a138a,"[2026, 8, 27, 14, 42, 26]"
9,村里界歷史圖資_113/Village_Sanhe.prj,147,128,294651ef,"[2026, 8, 27, 14, 42, 26]"


ZIP 全成員CRC通過: data/raw/113村里界.zip


,family,path,bytes,sha256,status
0,historical_boundaries,data/raw/114村里界.zip,22200490,08d2feeced98473215703defdbe9da8f8d551f2fb221029557b813e1a3dd8845,present


... (45 outputs left out)

,field,dtype,missing,distinct,definition
0,VILLCODE,str,0,1,來源官方村里代碼字串，透過既有crosswalk人口碼對照CPTVID
1,COUNTYNAME,str,0,1,來源縣市名稱
2,TOWNNAME,str,0,1,來源鄉鎮市區名稱
3,VILLNAME,str,0,1,來源村里名稱，保留原字形
4,VILLENG,str,0,1,來源村里英文名
5,COUNTYID,str,0,1,來源縣市識別碼
6,COUNTYCODE,str,0,1,來源縣市代碼原字串
7,TOWNID,str,0,1,來源鄉鎮市區識別碼
8,TOWNCODE,str,0,1,來源行政區代碼，保留原格式，不任意截取
9,NOTE,object,1,1,來源備註


**數值摘要（不把類別碼大小當實質尺度）**

,index,count,mean,std,min,25%,50%,75%,max
0,_source_row,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0


原始屬性完整列重複（不含診斷追溯欄） 0


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
